<a href="https://colab.research.google.com/github/RyuZULi/Python_AI/blob/master/260720.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
import urllib.request as req
url ="https://github.com/Elwing-Chou/ximen_ai_20260706/raw/refs/heads/main/titanic/train.csv"
req.urlretrieve(url,"train.csv")
url = "https://github.com/Elwing-Chou/ximen_ai_20260706/raw/refs/heads/main/titanic/test.csv"
req.urlretrieve(url,"test.csv")


('test.csv', <http.client.HTTPMessage at 0x7a5bb3a8b9e0>)

In [91]:
import pandas as pd
train_df = pd.read_csv("train.csv", encoding= "utf-8")
test_df = pd.read_csv("test.csv", encoding= "utf-8")

In [92]:
# 1. 過濾: 保留需要的
# 2. 轉換: old value => new value

# 我懶得分開做處理，處理X時我通常會把所有資料一起做
total_df = pd.concat([train_df,test_df])
total_df=total_df.drop(columns=["PassengerId","Survived"],axis=1)
total_df

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...
413,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [93]:
#  step 0 資料預處理(pre-processing)
def func(n):
    if pd.isna(n):
        return n
    else:
         return n.split(",")[-1].split(".")[0].strip() #strip?

# 強行覆蓋不要執行兩次，不然就全部執行
total_df["Name"] = total_df["Name"].apply(func)

In [94]:
dic = total_df["Ticket"].value_counts()

def func(n):
    if pd.isna(n):
        return n
    else:
         return dic[n]

total_df["Ticket"] = total_df["Ticket"].apply(func)
total_df["Ticket"]

,Ticket
0,1
1,2
2,1
3,2
4,1
...,...
413,1
414,3
415,1
416,1


In [95]:
total_df["Cabin"].value_counts()

def func(n):
    if pd.isna(n):
        return n
    else:
         return n[0]

total_df["Cabin"] = total_df["Cabin"].apply(func)
total_df

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,Mr,male,22.0,1,0,1,7.2500,NaN,S
1,1,Mrs,female,38.0,1,0,2,71.2833,C,C
2,3,Miss,female,26.0,0,0,1,7.9250,NaN,S
3,1,Mrs,female,35.0,1,0,2,53.1000,C,S
4,3,Mr,male,35.0,0,0,1,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...
413,3,Mr,male,NaN,0,0,1,8.0500,NaN,S
414,1,Dona,female,39.0,0,0,3,108.9000,C,C
415,3,Mr,male,38.5,0,0,1,7.2500,NaN,S
416,3,Mr,male,NaN,0,0,1,8.0500,NaN,S


In [96]:
# 資料預處理
# 1. 補缺失值: 補最可能的
# 2. One-Hot
# x 劃分兩種
# 1. 類別型(只有數種): Pclass, Name, Sex, Cabin, Embarked
# 2. 數值型(數字): Age, SibSp, Parch, Ticket, Fare
# 1. 類別最可能的值: 最常出現的
# 2. 數值: 中位數
# !!! 如果Pclass有缺, 該補最常出現(類別)
total_df.isna().sum()

,0
Pclass,0
Name,0
Sex,0
Age,263
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,1014
Embarked,2


In [97]:
# 如果要補最常出現
most = total_df["Embarked"].value_counts().idxmax()
total_df["Embarked"] = total_df["Embarked"].fillna(most)
total_df.isna().sum()

,0
Pclass,0
Name,0
Sex,0
Age,263
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,1014
Embarked,0


In [98]:
# 列出數字類型的欄位
col_num_fill =total_df.dtypes != "object"
# Pclass是類別型
col_nums = total_df.columns[col_num_fill].drop("Pclass")
#補中位數
med = total_df[col_nums].median()
total_df[col_nums] = total_df[col_nums].fillna(med)
total_df.isna().sum()

,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0
Cabin,1014
Embarked,0


In [99]:
# 類別型 onehotincoding: 把類別 => 多個是非題
# Q1 (如車子型號):欄位太多時，那是資料沒處理好
# Q2 唯一可以不做的時候: 二值類型類別(如:sex) => 決策數相關無腦one-hot也沒差
# Q3 同時是類別及數值(Pclass)做不做: 可作可不做 => 誰比較好?(一般情況隨便，要極致，自己比較結果)
# Q4 其實你不用去掉，如果你真要去掉(考慮美觀、使用者體驗)
# Q5 one-hot的另外好處 => 補類別缺失值(全部0 =>False)

# 如果真要丟掉
total_df["Name"].value_counts()


fil = total_df["Name"].value_counts() > 20
resered = total_df["Name"].value_counts().index[fil]
def func(n) :
    if n in resered :
        return n
    else :
        return None

total_df["Name"] = total_df["Name"].apply(func)

In [100]:
# show more col
pd.set_option('display.max_columns', 50)
total_df = pd.get_dummies(total_df)
total_df = pd.get_dummies(total_df, columns=["Pclass"])
total_df

,Age,SibSp,Parch,Ticket,Fare,Name_Master,Name_Miss,Name_Mr,Name_Mrs,Sex_female,Sex_male,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Embarked_C,Embarked_Q,Embarked_S,Pclass_1,Pclass_2,Pclass_3
0,22.0,1,0,1,7.2500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
1,38.0,1,0,2,71.2833,False,False,False,True,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False
2,26.0,0,0,1,7.9250,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True
3,35.0,1,0,2,53.1000,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,True,True,False,False
4,35.0,0,0,1,8.0500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,28.0,0,0,1,8.0500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
414,39.0,0,0,3,108.9000,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False
415,38.5,0,0,1,7.2500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True
416,28.0,0,0,1,8.0500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True


In [102]:
# 無腦: 1. 先補中位數 2. one-hot
# extra step: 利用欄位做出更有意義的欄位
total_df["Family"] = total_df["SibSp"] + total_df["Parch"]
total_df

,Age,SibSp,Parch,Ticket,Fare,Name_Master,Name_Miss,Name_Mr,Name_Mrs,Sex_female,Sex_male,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Embarked_C,Embarked_Q,Embarked_S,Pclass_1,Pclass_2,Pclass_3,Family
0,22.0,1,0,1,7.2500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,1
1,38.0,1,0,2,71.2833,False,False,False,True,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,1
2,26.0,0,0,1,7.9250,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,0
3,35.0,1,0,2,53.1000,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,True,True,False,False,1
4,35.0,0,0,1,8.0500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,28.0,0,0,1,8.0500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,0
414,39.0,0,0,3,108.9000,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,0
415,38.5,0,0,1,7.2500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,0
416,28.0,0,0,1,8.0500,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,0


In [ ]:
# 畫圖(看整體: 看這個問題是否真的可以被解決)
# 降維: 25cols => 2cols

# SNE 高維轉低維(高維空間倆兩的距離想辦法複製到低維空間)，且中型曲線加總後不變(面積都為1?)(梯度下降?)
# tSNE 刻意做出距離感，近的點近一點，遠的更遠一點